# A3.1 --- the node dial, as advantage over the boosting base

One figure: a panel per (dataset, regime), two curves per panel.

Each curve is the **paired difference of one boosting head against the base**, computed
fold by fold on the same held-out rows and then averaged over folds. The base is
therefore the zero line and is not drawn as a curve of its own --- there is nothing to
draw, since every point on the panel is already measured against it.

    cls+mol        the refined receptor REPLACES the raw protein vector
    cls+prot+mol   the refined receptor is ADDED beside it

**The question is not which alpha wins.** An argmax over a flat surface is noise. The
question is whether the surface is flat at all, which is what the fitted line asks and
what the box answers: a slope is fitted **per fold**, and the five slopes are the sample
the t-test is over. The fold is the unit of evidence here as everywhere else in this
project --- five model seeds agreeing on one fold are one observation, not five.

Two reductions happen before anything is drawn, in `alpha_grid`, and never here:
pair on (fold, seed), then average the seeds inside each fold. This notebook draws and
tests; it does not aggregate.


In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker
from scipy import stats

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.analysis import alpha_grid as ag
sys.path.insert(0, str(ROOT / "scripts" / "article_tables"))
import tablekit as tk          # for `holm` -- one implementation, not two

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/graph/v14_esm3_paper"  # the reported encoder regime
NODES      = "nodedial"     # the v9 NODE dial. "onehot" is the v8 gate and runs the
                            # other way -- the two must never share an axis
MOL_SOURCE = "chemberta"    # the dense grid exists only here; gin/ecfp hold alpha=1
SPLIT      = "test"         # "test" or "val". Nothing on this page is CHOSEN, so test
                            # is honest: alpha=1 is reported for what it means, not for
                            # being the argmax. Set "val" to see the same panels on the
                            # split a choice would have to be made on
HEADS      = ("cls+mol", "cls+prot+mol")   # the two curves, in legend order
SEED       = None           # model seed; None -> every one present

DATASETS   = ["m2or", "cc", "hc"]                 # columns, left to right
REGIMES    = ["transductive", "inductive"]        # rows, top to bottom
CI_LEVEL   = 0.95           # Student-t over FOLDS (seeds averaged inside each fold)
FIT        = True           # the least-squares line through the dial positions
SHOW_BOX   = True           # the slope test, per panel

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/alpha_dial"

# --------------------------- journal-facing knobs (24.09) --------------------
# Each is independent: switching any one to False leaves the others intact.
PANEL_LETTERS = True   # (a)-(f) above each panel, so text and caption can point at one
GRID_LABELS   = True   # dataset+metric as a COLUMN title, regime in the row's y label,
                       # instead of repeating both in six long panel titles
HOLM_IN_PANEL = True   # Holm-correct the two slope tests inside a panel. Twelve raw
                       # p-values on one figure is the first thing a referee counts
SLOPE_CI      = True   # print the slope with its 95% CI, not a bare p
BOX_CORNER    = "auto" # "auto" puts the slope box where the curves are not; or give
                       # "lower left" / "upper left" / "lower right" / "upper right"
OKABE_ITO     = True   # the colour-vision-safe pair (Wong 2011) instead of blue+green
FIT_DOTTED    = True   # dot the fit so it cannot be confused with the dashed base line
BAND_ALPHA    = 0.12   # two overlapping 95% bands at 0.16 read as mud
FIG_WIDTH_IN  = 7.1    # build at the FINAL printed width (180 mm). Fonts do not scale,
                       # so a figure shrunk into a column is a figure with 4.5 pt text
FIG_FORMATS   = ("png", "pdf")   # vector for print; PNG for looking at it here
# =============================================================================

FRAMES = {}
for head in HEADS:
    FRAMES[head] = ag.load(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
                           seed=SEED, split=SPLIT, combo=head)

DF = pd.concat(FRAMES.values(), ignore_index=True)   # the dial guard and the alpha list
ALPHAS = sorted(DF.loc[DF.arm == "gate", "alpha"].dropna().unique())

In [ ]:
# ---------------------------------------------------------------- what is on disk
# One dial per axis. On a v8 GATE run alpha mixes the graph's OUTPUT against a frozen ESM
# branch; on a v9 NODE dial it mixes the graph's INPUT and runs the other way (alpha=0 is
# receptor identity, alpha=1 the ordinary graph). A frame holding both would put opposite
# meanings on one x axis, so refuse it here rather than discover it in a plot.
dials = sorted(DF["nodes"].astype(str).unique())
if len(dials) != 1:
    raise SystemExit(f"this frame holds {dials} -- two dials cannot share an axis. "
                     f"Set NODES to one of them.")

print(f"{len(ALPHAS)} dial positions | dial: {dials[0]} | split: {SPLIT}")
print("alphas:", ", ".join(f"{a:g}" for a in ALPHAS))
for head, f in FRAMES.items():
    n = sorted(f.loc[f.arm == "gate", "alpha"].dropna().unique())
    print(f"  {head:<13} {len(f):5d} rows | {len(n)} alphas | "
          f"series: {', '.join(sorted(f.series.unique()))}")
# Coverage on ONE head: the two frames hold the same cells, and concatenating them
# would report twice the rows per series for no reason.
ag.coverage(FRAMES[HEADS[0]])

In [ ]:
MOF = ag.metric_for(DF)          # metric of record per dataset: AUROC / R2
DS_LABEL = {"m2or": "M2OR", "cc": "Mosquito (Carey)", "hc": "Fly (Hallem-Carlson)"}
RG_LABEL = {"transductive": "Transductive", "inductive": "Cold molecule"}
print("metric of record:", MOF)

## The battery

Rows are regimes, columns are datasets. Zero is the boosting base on the same folds.


In [ ]:
# =============================== THE FIGURE ==================================
HEAD_C = ({"cls+mol": fk.CRIT["coverage"], "cls+prot+mol": fk.CRIT["balance_bits"]}
          if OKABE_ITO else
          {"cls+mol": fk.C["gate"], "cls+prot+mol": fk.C["graph_legacy"]})
HEAD_M = {"cls+mol": "o", "cls+prot+mol": "s"}
FIT_DASH = (0, (1, 1.6)) if FIT_DOTTED else (0, (4, 1.5))
METRIC_TEX = {"R2": "$R^2$"}
LETTERS = "abcdefghijklmnop"


def slope_per_fold(pf, alphas):
    """One least-squares slope of the advantage on alpha, PER FOLD.

    The fold is the sample. A slope fitted on the (fold, seed) rows would treat five
    seeds that share a held-out set as five observations, which is the error this whole
    project is organised against."""
    out = {}
    for fold, g in pf.groupby("fold"):
        g = g[g.alpha.isin(alphas)].sort_values("alpha")
        if len(g) >= 3 and g.alpha.nunique() >= 3:
            out[int(fold)] = float(np.polyfit(g.alpha.astype(float),
                                              g.delta.astype(float), 1)[0])
    return pd.Series(out, dtype=float)


def slope_test(slopes):
    """mean slope, its t interval over folds, and the two-sided p against zero."""
    s = slopes.dropna()
    if len(s) < 3 or float(s.std(ddof=1)) == 0:
        return dict(n=len(s), slope=float(s.mean()) if len(s) else np.nan,
                    lo=np.nan, hi=np.nan, p=np.nan)
    m, hw, n = ag.ci(list(s), CI_LEVEL)
    return dict(n=n, slope=m, lo=m - hw, hi=m + hw,
                p=float(stats.ttest_1samp(s, 0.0).pvalue))


def box_corner(ax, ys):
    """Where the curves are not. A box that lands on the data is worse than no box."""
    if BOX_CORNER != "auto":
        return BOX_CORNER
    lo, hi = ax.get_ylim()
    return "upper left" if np.nanmean(ys) < (lo + hi) / 2 else "lower left"


panel_w = (FIG_WIDTH_IN / len(DATASETS)) if FIG_WIDTH_IN else 3.5
fig, axes = plt.subplots(len(REGIMES), len(DATASETS), squeeze=False,
                         figsize=(panel_w * len(DATASETS), panel_w * 0.84 * len(REGIMES)))
rows = []
for r, regime in enumerate(REGIMES):
    for c, ds in enumerate(DATASETS):
        a = axes[r][c]
        metric = MOF.get(ds)
        drawn, stats_here, ys = False, [], []
        for head in HEADS:
            f = FRAMES[head]
            sub = f[(f.dataset == ds) & (f.regime == regime)]
            if sub.empty or metric is None:
                continue
            d = ag.delta_vs(sub, metric, ref_arm="boost_full", by=("series",),
                            level=CI_LEVEL)
            if not len(d):
                continue
            drawn |= fk.curve_with_band(a, d, HEAD_C[head], marker=HEAD_M[head],
                                        band_alpha=BAND_ALPHA)
            ys += list(pd.to_numeric(d["mean"], errors="coerce").dropna())
            pf = ag.delta_folds(sub, metric, ref_arm="boost_full", by=("series",))
            st = slope_test(slope_per_fold(pf, ALPHAS))
            st.update(dataset=ds, regime=regime, head=head, metric=metric)
            stats_here.append(st)
            if FIT and np.isfinite(st["slope"]):
                x = np.array([min(ALPHAS), max(ALPHAS)], dtype=float)
                y0 = float(d.sort_values("alpha")["mean"].mean())
                a.plot(x, y0 + st["slope"] * (x - float(np.mean(ALPHAS))),
                       color=HEAD_C[head], lw=1.1, ls=FIT_DASH, zorder=1.5)

        # Holm over the tests INSIDE this panel: the family a reader reads together.
        adj = tk.holm({s["head"]: s["p"] for s in stats_here}) if HOLM_IN_PANEL else {}
        for s in stats_here:
            s["p_shown"] = float(adj.get(s["head"], s["p"])) if HOLM_IN_PANEL else s["p"]
        rows += stats_here

        a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"],
                  zorder=2.5)
        if not drawn:
            fk.note_empty(a)
            continue
        fk.alpha_axis(a, ALPHAS)

        if SHOW_BOX and stats_here:
            items = [TextArea("slope per fold" + (" (Holm)" if HOLM_IN_PANEL else ""),
                              textprops=dict(size=6.4, color=fk.MUTED))]
            for s in stats_here:
                txt = f"{s['head']}: {s['slope']:+.3f}"
                if SLOPE_CI and np.isfinite(s["lo"]):
                    txt += f" [{s['lo']:+.3f}, {s['hi']:+.3f}]"
                txt += (f"  p={s['p_shown']:.3f}" if np.isfinite(s["p_shown"])
                        else "  p=n/a")
                items.append(TextArea(txt, textprops=dict(size=6.5,
                                                          color=HEAD_C[s["head"]])))
            box = AnchoredOffsetbox(loc=box_corner(a, ys), frameon=True, borderpad=0.35,
                                    pad=0.3, child=VPacker(children=items, sep=2, pad=0))
            box.patch.set(facecolor="white", edgecolor="#c7ccd1", linewidth=0.7,
                          alpha=0.92)
            a.add_artist(box)

        mtex = METRIC_TEX.get(metric, metric)
        if GRID_LABELS:
            if r == 0:
                a.set_title(f"{DS_LABEL.get(ds, ds)}  ({mtex})", fontsize=9)
            if c == 0:
                a.set_ylabel(f"{RG_LABEL.get(regime, regime)}\nadvantage over the base")
        else:
            a.set_title(f"{DS_LABEL.get(ds, ds)} - {RG_LABEL.get(regime, regime)}"
                        f"  ({mtex})", fontsize=8.5)
            if c == 0:
                a.set_ylabel("advantage over the base")
        if PANEL_LETTERS:
            a.annotate(f"({LETTERS[r * len(DATASETS) + c]})", xy=(0, 1),
                       xycoords="axes fraction", xytext=(-32, 4),
                       textcoords="offset points", fontweight="bold", fontsize=9,
                       ha="left", va="bottom")
        if r == len(REGIMES) - 1:
            a.set_xlabel(r"$\alpha$")

handles = [Line2D([], [], color=HEAD_C[h], marker=HEAD_M[h], label=h) for h in HEADS]
handles += [Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
                   label="the base (zero)"),
            Line2D([], [], color=fk.MUTED, ls=FIT_DASH, lw=1.1,
                   label="least-squares fit"),
            Line2D([], [], color=fk.MUTED, lw=6, alpha=BAND_ALPHA * 1.6,
                   label=f"band = {CI_LEVEL:.0%} CI over folds")]
fk.figlegend(fig, handles, ncol=5)
fk.savefig(fig, "alpha_dial_delta", FIG_DIR, SAVE_FIGS, formats=FIG_FORMATS)
plt.show()

SLOPES = pd.DataFrame(rows)
display(SLOPES.round(4))